In [1]:
from langchain.chat_models import ChatOpenAI
from langchain.memory import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate, FewShotChatMessagePromptTemplate, MessagesPlaceholder
from langchain.schema.runnable import RunnablePassthrough, RunnableLambda

# 1. 모델 설정
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 2. Few-Shot 예시 프롬프트 구성
examples = [
    {"input": "탑건", "output": "🛩️👨‍✈️🔥"},
    {"input": "대부", "output": "👨‍👨‍👦🔫🍝"},
]

example_prompt = ChatPromptTemplate.from_messages([
    ("human", "{input}"),
    ("ai", "{output}")
])

few_shot_prompt = FewShotChatMessagePromptTemplate(
    example_prompt=example_prompt,
    examples=examples,
)

# 3. 메모리 설정
memory = ConversationBufferMemory(
    memory_key="history", 
    return_messages=True
)

# 4. 프롬프트 구성
final_prompt = ChatPromptTemplate.from_messages([
    ("system", """
사용자가 영화 제목을 입력하면 정확히 3개의 이모티콘으로만 답하라.
사용자가 이전 대화 내용을 물어보면 대화 기록을 바탕으로 자연어로 답하라.
"""),
    few_shot_prompt,
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}"),
])

# 5. LCEL 체인 설계
# 메모리에서 history를 불러와 프롬프트에 전달하는 구조
chain = (
    RunnablePassthrough.assign(        
        history=RunnableLambda(
            lambda _: memory.load_memory_variables({})["history"]
        )
    )
    | final_prompt
    | model
)

def ask_movie(user_input):
    # 체인 실행
    result = chain.invoke({"input": user_input})
    # 대화 내용을 메모리에 수동으로 저장
    memory.save_context(
        {"input": user_input}, 
        {"output": result.content}
    )
    print(result)


In [2]:
ask_movie("타이타닉")
ask_movie("인터스텔라")

content='🚢💔🌊'
content='🌌🚀⏳'


In [3]:
ask_movie("내가 먼저 질문한 영화가 뭐였어?")

content='당신이 처음 질문한 영화는 "탑건"이었습니다.'
